# Three Neuron Group



In [6]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import nest
import nest.raster_plot
import time

In [7]:

def log_parameters(N, Ne, Ni1, Ni2, neuron_data, connection_matrix, input_strength, bottom_up_input, top_down_input):
    print("Simulation Parameters:")
    print(f"Total neurons: {N}")
    print(f"Excitatory neurons: {Ne}")
    print(f"Inhibitory group 1 neurons: {Ni1}")
    print(f"Inhibitory group 2 neurons: {Ni2}")
    print("\nNeuron Data:")
    print(neuron_data)
    print("\nConnection Matrix:")
    print(connection_matrix)
    print("\nInput Strength:")
    print(input_strength)
    print("\nBottom-up Input:")
    print(bottom_up_input)
    print("\nTop-down Input:")
    print(top_down_input)

In [8]:
import numpy as np
import pandas as pd
import nest
import time

import matplotlib.pyplot as plt

# Define parameters
N = 100  # Total number of neurons
Ne_ratio = 0.8  # Ratio of excitatory neurons
Ni1_ratio = 0.1  # Ratio of first group of inhibitory neurons
Ni2_ratio = 0.1  # Ratio of second group of inhibitory neurons

# Calculate the number of neurons based on the ratios
Ne = int(N * Ne_ratio)
Ni1 = int(N * Ni1_ratio)
Ni2 = int(N * Ni2_ratio)

# Create neuron data DataFrame
neuron_data = pd.DataFrame(index=np.arange(N))
neuron_data['type'] = ['excitatory'] * Ne + ['inhibitory_1'] * Ni1 + ['inhibitory_2'] * Ni2

# Assign different parameters to each neuron type
# Parameters for excitatory neurons
neuron_data.loc[neuron_data['type'] == 'excitatory', 'tau_m'] = 20.0
neuron_data.loc[neuron_data['type'] == 'excitatory', 'cm'] = 0.2
neuron_data.loc[neuron_data['type'] == 'excitatory', 'v_rest'] = -65.0
neuron_data.loc[neuron_data['type'] == 'excitatory', 'v_reset'] = -65.0
neuron_data.loc[neuron_data['type'] == 'excitatory', 'v_thresh'] = -50.0
neuron_data.loc[neuron_data['type'] == 'excitatory', 'e_rev_E'] = 0.0
neuron_data.loc[neuron_data['type'] == 'excitatory', 'e_rev_I'] = -80.0
neuron_data.loc[neuron_data['type'] == 'excitatory', 'tau_syn_E'] = 5.0
neuron_data.loc[neuron_data['type'] == 'excitatory', 'tau_syn_I'] = 10.0
neuron_data.loc[neuron_data['type'] == 'excitatory', 'a'] = 1
neuron_data.loc[neuron_data['type'] == 'excitatory', 'b'] = 0.02
neuron_data.loc[neuron_data['type'] == 'excitatory', 'tau_w'] = 600.0
neuron_data.loc[neuron_data['type'] == 'excitatory', 'i_offset'] = 4 * np.random.rand(Ne)

# Parameters for first group of inhibitory neurons
neuron_data.loc[neuron_data['type'] == 'inhibitory_1', 'tau_m'] = 10.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_1', 'cm'] = 0.1
neuron_data.loc[neuron_data['type'] == 'inhibitory_1', 'v_rest'] = -70.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_1', 'v_reset'] = -65.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_1', 'v_thresh'] = -50.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_1', 'e_rev_E'] = 0.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_1', 'e_rev_I'] = -75.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_1', 'tau_syn_E'] = 3.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_1', 'tau_syn_I'] = 8.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_1', 'a'] = 1
neuron_data.loc[neuron_data['type'] == 'inhibitory_1', 'b'] = 0.015
neuron_data.loc[neuron_data['type'] == 'inhibitory_1', 'tau_w'] = 600.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_1', 'i_offset'] = 2 * np.random.rand(Ni1)

# Parameters for second group of inhibitory neurons
neuron_data.loc[neuron_data['type'] == 'inhibitory_2', 'tau_m'] = 16.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_2', 'cm'] = 0.15
neuron_data.loc[neuron_data['type'] == 'inhibitory_2', 'v_rest'] = -68.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_2', 'v_reset'] = -65.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_2', 'v_thresh'] = -52.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_2', 'e_rev_E'] = 0.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_2', 'e_rev_I'] = -78.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_2', 'tau_syn_E'] = 4.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_2', 'tau_syn_I'] = 9.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_2', 'a'] = 0.003
neuron_data.loc[neuron_data['type'] == 'inhibitory_2', 'b'] = 0.018
neuron_data.loc[neuron_data['type'] == 'inhibitory_2', 'tau_w'] = 600.0
neuron_data.loc[neuron_data['type'] == 'inhibitory_2', 'i_offset'] = 2 * np.random.rand(Ni2)

# Define a 3x3 connection matrix indicating connection probabilities between groups
# Rows represent outputs and columns represent inputs: [excitatory, inhibitory_1, inhibitory_2]
connection_matrix = np.array([
    [0.02, 0.01, 0.01],  # Outputs from excitatory neurons
    [0.04, 0.02, 0.01],  # Outputs from inhibitory_1 neurons
    [0.04, 0.01, 0.02]   # Outputs from inhibitory_2 neurons
])

# Define input strength for each neuron group
input_strength = {
    'excitatory': 2.0,
    'inhibitory_1': 4.0,
    'inhibitory_2': 5.0
}

# Define bottom-up and top-down input vectors
bottom_up_input = np.array([0.12178222, 0.61389708, 0.53952226, 0.05811071, 0.55009501,
       0.68675795, 0.81001471, 0.21750756, 0.78141485, 0.23565596,
       0.75766387, 0.02034385, 0.65309662, 0.9915203 , 0.42683985,
       0.37224836, 0.90353016, 0.00635595, 0.19878682, 0.9464665 ,
       0.23117152, 0.30105121, 0.85901979, 0.14425687, 0.29849147,
       0.76590299, 0.09131473, 0.32044681, 0.84297596, 0.60225067,
       0.67224833, 0.23752574, 0.9221909 , 0.2735787 , 0.43026112,
       0.38019239, 0.87305989, 0.12004674, 0.39026437, 0.65028295,
       0.39781851, 0.69361214, 0.09994601, 0.87394779, 0.76871501,
       0.65199255, 0.75504   , 0.53035842, 0.66921239, 0.60298276,
       0.02329587, 0.32881746, 0.4652889 , 0.94996061, 0.85764358,
       0.99471237, 0.47248013, 0.67105282, 0.8665911 , 0.23210561,
       0.2662184 , 0.94133726, 0.82126536, 0.97669991, 0.45848655,
       0.38226917, 0.05139013, 0.12613202, 0.27135841, 0.58080523,
       0.0354074 , 0.40039338, 0.79836351, 0.21071637, 0.12216132,
       0.71321129, 0.96396164, 0.26596041, 0.31842385, 0.32709393,
       0.3078423 , 0.32079604, 0.49807122, 0.44413018, 0.50297744,
       0.51638233, 0.81353067, 0.13943993, 0.11527574, 0.38912456,
       0.91005365, 0.4013345 , 0.28179842, 0.51263778, 0.03191559,
       0.375341  , 0.28220119, 0.66510638, 0.09345107, 0.79475866])

top_down_input = np.array([0.37703602, 0.27093614, 0.06717243, 0.42555134, 0.90857397,
       0.44159152, 0.17144098, 0.95735263, 0.87415192, 0.93389023,
       0.09348735, 0.95215277, 0.97317488, 0.07735105, 0.60432707,
       0.26723632, 0.43695251, 0.31988982, 0.18724177, 0.33864615,
       0.74186307, 0.30313344, 0.52658208, 0.16117122, 0.15878648,
       0.95299456, 0.83236143, 0.04125933, 0.18599133, 0.0579066 ,
       0.43483902, 0.30120935, 0.30790283, 0.30092474, 0.37699992,
       0.67887586, 0.46983354, 0.88211634, 0.78838873, 0.44694089,
       0.14193269, 0.38983946, 0.26646577, 0.72676865, 0.75443631,
       0.77285071, 0.4467232 , 0.6505019 , 0.89077995, 0.15099425,
       0.24860667, 0.10373581, 0.60617003, 0.31515047, 0.69245157,
       0.89516296, 0.93385764, 0.76409696, 0.02942408, 0.71306555,
       0.69830826, 0.37751298, 0.59176065, 0.26984758, 0.92356199,
       0.56992231, 0.72021973, 0.17627563, 0.25110247, 0.70915338,
       0.41419847, 0.5732014 , 0.24284173, 0.62376655, 0.44399377,
       0.33321465, 0.82650826, 0.34865863, 0.294389  , 0.65691923,
       0.37756248, 0.35929491, 0.56387054, 0.37755731, 0.07871262,
       0.36403989, 0.51453968, 0.94654308, 0.42943899, 0.4578002 ,
       0.56298874, 0.04275855, 0.06891849, 0.63224322, 0.18840641,
       0.66831373, 0.81526073, 0.61701146, 0.87658505, 0.87329295])

# Log parameters
def log_parameters(N, Ne, Ni1, Ni2, neuron_data, connection_matrix, input_strength, bottom_up_input, top_down_input):
    print(f"N: {N}, Ne: {Ne}, Ni1: {Ni1}, Ni2: {Ni2}")
    print("Neuron data:")
    print(neuron_data.head())
    print("Connection matrix:")
    print(connection_matrix)
    print("Input strength:")
    print(input_strength)
    print("Bottom-up input:")
    print(bottom_up_input)
    print("Top-down input:")
    print(top_down_input)

# Run simulation
def run_simulation(neuron_data, connection_matrix, input_strength, bottom_up_input, top_down_input, runtime=1000.0, plot=False, num_threads=1):
    N = len(neuron_data)
    Ne = len(neuron_data[neuron_data['type'] == 'excitatory'])
    Ni1 = len(neuron_data[neuron_data['type'] == 'inhibitory_1'])
    Ni2 = len(neuron_data[neuron_data['type'] == 'inhibitory_2'])
    
    log_parameters(N, Ne, Ni1, Ni2, neuron_data, connection_matrix, input_strength, bottom_up_input, top_down_input)
    
    nest.ResetKernel()
    nest.SetKernelStatus({'local_num_threads': num_threads})
    
    # Create neuron populations with NEST
    neurons = nest.Create('aeif_cond_exp', N)
    
    for i, neuron in enumerate(neurons):
        neuron_params = {
            'tau_m': neuron_data['tau_m'].values[i],
            'C_m': neuron_data['cm'].values[i],
            'E_L': neuron_data['v_rest'].values[i],
            'V_reset': neuron_data['v_reset'].values[i],
            'V_th': neuron_data['v_thresh'].values[i],
            'E_ex': neuron_data['e_rev_E'].values[i],
            'E_in': neuron_data['e_rev_I'].values[i],
            'tau_syn_ex': neuron_data['tau_syn_E'].values[i],
            'tau_syn_in': neuron_data['tau_syn_I'].values[i],
            'a': neuron_data['a'].values[i],
            'b': neuron_data['b'].values[i],
            'tau_w': neuron_data['tau_w'].values[i],
            'I_e': neuron_data['i_offset'].values[i] + bottom_up_input[i] + top_down_input[i],
        }
        print(f"Setting parameters for neuron {i}: {neuron_params}")
        nest.SetStatus([neuron], neuron_params)
    
    # Create synapses based on connection matrix
    groups = [
        neuron_data[neuron_data['type'] == 'excitatory'].index,
        neuron_data[neuron_data['type'] == 'inhibitory_1'].index,
        neuron_data[neuron_data['type'] == 'inhibitory_2'].index
    ]
    
    for i, pre_group in enumerate(groups):
        for j, post_group in enumerate(groups):
            if connection_matrix[i, j] > 0:
                conn_dict = {'rule': 'fixed_indegree', 'indegree': int(connection_matrix[i, j] * len(pre_group))}
                syn_dict = {'weight': input_strength[neuron_data.iloc[pre_group[0]]['type']], 'delay': 1.0}
                nest.Connect(neurons[pre_group.to_list()], neurons[post_group.to_list()], conn_dict, syn_dict)
    
    # Set up recording
    spike_recorder = nest.Create('spike_recorder')
    nest.Connect(neurons, spike_recorder)
    
    # Run the simulation
    start_time = time.time()
    nest.Simulate(runtime)
    end_time = time.time()
    
    # Get recorded data
    spikes = nest.GetStatus(spike_recorder, 'events')[0]
    
    # Log simulation time
    print(f"Simulation runtime: {end_time - start_time} seconds")

    if plot:
        nest.raster_plot.from_device(spike_recorder)
        plt.show()

    return spikes

# Example usage
results = run_simulation(neuron_data, connection_matrix, input_strength, bottom_up_input, top_down_input, runtime=1000.0, plot=True)


N: 100, Ne: 80, Ni1: 10, Ni2: 10
Neuron data:
         type  tau_m   cm  v_rest  v_reset  v_thresh  e_rev_E  e_rev_I  \
0  excitatory   20.0  0.2   -65.0    -65.0     -50.0      0.0    -80.0   
1  excitatory   20.0  0.2   -65.0    -65.0     -50.0      0.0    -80.0   
2  excitatory   20.0  0.2   -65.0    -65.0     -50.0      0.0    -80.0   
3  excitatory   20.0  0.2   -65.0    -65.0     -50.0      0.0    -80.0   
4  excitatory   20.0  0.2   -65.0    -65.0     -50.0      0.0    -80.0   

   tau_syn_E  tau_syn_I    a     b  tau_w  i_offset  
0        5.0       10.0  1.0  0.02  600.0  3.338020  
1        5.0       10.0  1.0  0.02  600.0  1.489168  
2        5.0       10.0  1.0  0.02  600.0  3.845512  
3        5.0       10.0  1.0  0.02  600.0  3.661602  
4        5.0       10.0  1.0  0.02  600.0  3.317684  
Connection matrix:
[[0.02 0.01 0.01]
 [0.04 0.02 0.01]
 [0.04 0.01 0.02]]
Input strength:
{'excitatory': 2.0, 'inhibitory_1': 4.0, 'inhibitory_2': 5.0}
Bottom-up input:
[0.12178222 0.61

TypeError: 'nodes' must be NodeCollection or a SynapseCollection.